In [1]:
from pathlib import Path
import os,ast,json,hashlib
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import numpy as np,h5py,torch
from torch import nn
from sklearn.metrics import roc_auc_score,average_precision_score
ROOT=next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p/'dataset_2.csv').is_file());RUN=ROOT/'fusion/cnn_bilstm_attention_v1';OUT=RUN/'codex_independent_audit';V3=ROOT/'cnn/cnn wave/eew_causal_v3';CR=V3/'cnn_eew_training';BR=ROOT/'bilstm_eew/bilstm_training_v1';TAB=ROOT/'bilstm_eew/causal_tabular_v1/tabular_features.h5'
DEV=torch.device('cuda' if torch.cuda.is_available() else 'cpu');torch.set_num_threads(4);torch.use_deterministic_algorithms(True);torch.backends.cudnn.benchmark=False
def sha(p):
 h=hashlib.sha256()
 with p.open('rb') as f:
  for b in iter(lambda:f.read(4194304),b''):h.update(b)
 return h.hexdigest()

FUS=ROOT/'fusion/cnn_bilstm_v1'
cks={h:torch.load(RUN/'models'/f'attention_{h}s_best.pt',map_location='cpu',weights_only=False) for h in [1,2,3]}
paths=[Path(p) for p in cks[1]['input_hashes']]+[RUN/n for n in ['final_attention_report.md','01_attention_training.ipynb','02_attention_evaluation.ipynb']]+[RUN/'models'/f'attention_{h}s_best.pt' for h in [1,2,3]]
before={str(p):sha(p) for p in paths}
reported=json.loads((RUN/'final_attention_report.md').read_text(encoding='utf-8').split('```json\n')[1].split('\n```')[0])
training=json.loads((RUN/'01_attention_training.ipynb').read_text());evaluation=json.loads((RUN/'02_attention_evaluation.ipynb').read_text())
R={'checks':{},'horizons':{},'input_hashes':before}
def check(k,ok,evidence):R['checks'][k]={'status':'PASS' if ok else 'FAIL','evidence':evidence}
# Only exact architecture declarations imported; audit independently implements embeddings, pairing, inference and metrics.
for path,cell,name in [(CR/'01_cnn_training.ipynb',1,'EarlyCNN'),(BR/'01_bilstm_training.ipynb',0,'TabularBiLSTM')]:
 tree=ast.parse(''.join(json.loads(path.read_text(encoding='utf-8'))['cells'][cell]['source']));node=next(n for n in tree.body if isinstance(n,ast.ClassDef) and n.name==name);exec(compile(ast.Module(body=[node],type_ignores=[]),'class_definition_only','exec'))
class Head(nn.Module):
 def __init__(self):
  super().__init__();self.shared=nn.Sequential(nn.Linear(64,32),nn.SiLU(),nn.Dropout(.2));self.detection=nn.Linear(32,1);self.magnitude=nn.Linear(32,1)
 def forward(self,x):
  x=self.shared(x);return self.detection(x)[:,0],self.magnitude(x)[:,0]
def detection(y,p,t=.5):
 y=np.asarray(y)==1;q=p>=t;tp=int((y&q).sum());tn=int((~y&~q).sum());fp=int((~y&q).sum());fn=int((y&~q).sum())
 return dict(accuracy=(tp+tn)/len(y),precision=tp/(tp+fp) if tp+fp else 0.,recall=tp/(tp+fn),f1=2*tp/(2*tp+fp+fn),false_positive_rate=fp/(fp+tn),roc_auc=float(roc_auc_score(y,p)),pr_auc_ap=float(average_precision_score(y,p)),confusion_matrix=[[tn,fp],[fn,tp]])
def magnitude(a,p):
 a=a.astype(float);p=p.astype(float);e=p-a;s=a>=5
 return dict(mae=float(abs(e).mean()),rmse=float(np.sqrt((e**2).mean())),r2=float(1-(e**2).sum()/((a-a.mean())**2).sum()),bias=float(e.mean()),actual_variance=float(a.var()),predicted_variance=float(p.var()),std_ratio=float(p.std()/a.std()),slope=float(np.polyfit(a,p,1)[0]),strong_n=int(s.sum()),strong_underprediction_percent=float((e[s]<0).mean()*100),strong_bias=float(e[s].mean()))
# Independent parent-key join and metadata checks; no regeneration/full-array causality audit.
MAP={};parents={}
with h5py.File(TAB,'r') as tf:
 for split in ['train','validation','test']:
  with h5py.File(V3/'dataset'/f'eew_dataset_{split}.h5','r') as wf:
   w=wf['parent_id'].asstr()[:];t=tf[split]['parent_id'].asstr()[:];assert len(set(w))==len(w) and len(set(t))==len(t) and set(w)==set(t),'Critical parent join failure'
   lookup={p:i for i,p in enumerate(t)};ix=np.array([lookup[p] for p in w]);MAP[split]=ix;parents[split]=set(w);mismatches=[]
   for key in ['source_id','parent_id','pair_id','combined_index','labels','magnitude','magnitude_mask']+[f'{k}_{h}s' for h in [1,2,3] for k in ['classification_mask','magnitude_mask','pre_s_valid']]:
    a=wf[key].asstr()[:] if wf[key].dtype.kind=='O' else wf[key][:];ds=tf[split][key];b=ds.asstr()[:] if ds.dtype.kind=='O' else ds[:]
    if not np.array_equal(a,b[ix]):mismatches.append(key)
   check(split+'_pairing',not mismatches,{'records':len(w),'mismatches':mismatches,'permutation_nonidentity_rows':int((ix!=np.arange(len(w))).sum())})
   for h in [1,2,3]:
    y=wf['labels'][:];reg=wf[f'magnitude_mask_{h}s'][:];keep=wf[f'classification_mask_{h}s'][:];pre=(y==1)&(wf['reference_sample'][:]+h*100<wf['s_arrival_sample'][:])
    check(f'{split}_{h}s_masks_cutoff',np.array_equal(reg,pre) and np.array_equal(keep,pre|(y==0)) and np.all(wf['magnitude'][:][y==0]==-1) and wf[f'windows_{h}s_norm'].shape[1]==100+h*100 and tf[split][f'{h}s/X'].shape[1:]==(10+h*10,7),'Labels/first-S eligibility/noise sentinel; only observed horizon enters models.')
    sg=tf[f'scalers/{h}s'];check(f'{split}_{h}s_feature_scaling_metadata',sg.attrs['fit_split']=='train' and sg.attrs['fit_records']==int(tf[f'train/classification_mask_{h}s'][:].sum()),'Approved precomputed feature scaler provenance reused; feature extraction not rerun.')
for a,b in [('train','validation'),('train','test'),('validation','test')]:check(a+'/'+b+'_parent_isolation',not(parents[a]&parents[b]),len(parents[a]&parents[b]))
@torch.inference_mode()
def encode(split,h,c,b,cc,bc,common=False):
 with h5py.File(V3/'dataset'/f'eew_dataset_{split}.h5','r') as wf,h5py.File(TAB,'r') as tf:
  mask=wf[f'classification_mask_{h}s'][:].astype(bool)
  if common:mask=np.logical_and.reduce([wf[f'classification_mask_{j}s'][:] for j in [1,2,3]])
  x=wf[f'windows_{h}s_norm'][:][mask,None,:];tab=tf[split][f'{h}s/X'][:][MAP[split]][mask];y=wf['labels'][:][mask];m=wf['magnitude'][:][mask];reg=wf[f'magnitude_mask_{h}s'][:][mask];ids=wf['source_id'].asstr()[:][mask];pid=wf['parent_id'].asstr()[:][mask]
 states=[];em=[];cp=[];cm=[];bp=[];bm=[]
 for i in range(0,len(x),256):
  xx=torch.from_numpy(x[i:i+256]).to(DEV);tt=torch.from_numpy(tab[i:i+256]).to(DEV)
  u=c.features(xx.sign()*xx.abs().log1p());ce=c.shared(torch.cat((u.mean(-1),u.amax(-1)),1));v,_=b.lstm(tt);states.append(v.cpu());be=b.shared(v.mean(1));em.append(torch.cat((ce,be),1).cpu())
  cp.append(c.detection(ce).flatten().sigmoid().cpu());cm.append(c.magnitude(ce).flatten().cpu());bp.append(b.detect(be).flatten().sigmoid().cpu());bm.append(b.mag(be).flatten().cpu())
 return dict(states=torch.cat(states),z=torch.cat(em),y=y,m=m,reg=reg,ids=ids,parent=pid,cp=torch.cat(cp).numpy(),cm=torch.cat(cm).numpy()*cc['target_scale']+cc['target_center'],bp=torch.cat(bp).numpy(),bm=torch.cat(bm).numpy()*bc['scale']+bc['center'])
def uncertainty(y,target,fp,fm,bp,bm):
 rng=np.random.default_rng(2026);eq=np.flatnonzero(y==1);noise=np.flatnonzero(y==0);df=[];dm=[]
 def f1(y,p):
  a=y==1;b=p>=.5;tp=(a&b).sum();return 2*tp/(2*tp+(~a&b).sum()+(a&~b).sum())
 for _ in range(500):
  e=rng.choice(eq,len(eq),replace=True);n=rng.choice(noise,len(noise),replace=True);ix=np.r_[e,n];df.append(float(f1(y[ix],fp[ix])-f1(y[ix],bp[ix])));dm.append(float((abs(fm[e]-target[e])-abs(bm[e]-target[e])).mean()))
 return {'f1_difference_ci95':np.quantile(df,[.025,.975]).tolist(),'mae_difference_ci95':np.quantile(dm,[.025,.975]).tolist()}

# Independent functional attention/head implementation: no original Attention forward or inference function executed.
@torch.inference_mode()
def attention(d,ck):
 st={k:v.to(DEV) for k,v in ck['head_state'].items()};logits=[];mags=[];weights=[]
 def linear(x,name):return nn.functional.linear(x,st[name+'.weight'],st.get(name+'.bias'))
 for i in range(0,len(d['y']),256):
  c=((d['z'][i:i+256,:32]-ck['cmean'])/ck['cscale']).to(DEV);s=((d['states'][i:i+256]-ck['smean'])/ck['sscale']).to(DEV)
  scores=linear(torch.tanh(linear(s,'attention.0')),'attention.2').squeeze(-1);w=torch.softmax(scores,dim=1)
  joined=torch.cat((c,(w.unsqueeze(-1)*s).sum(dim=1)),dim=1);hidden=nn.functional.silu(linear(joined,'shared.0'))
  logits.append(linear(hidden,'detection').flatten().cpu());mags.append(linear(hidden,'magnitude').flatten().cpu());weights.append(w.cpu())
 return torch.cat(logits),torch.cat(mags),torch.cat(weights).numpy()
@torch.inference_mode()
def fusion(d,ck):
 head=Head().to(DEV);head.load_state_dict(ck['head_state']);head.eval();ps=[];ms=[]
 for z in ((d['z']-ck['embedding_mean'])/ck['embedding_scale']).split(256):
  p,m=head(z.to(DEV));ps.append(p.sigmoid().cpu());ms.append(m.cpu())
 return torch.cat(ps).numpy(),torch.cat(ms).numpy()*ck['target_scale']+ck['target_center']
for h,ck in cks.items():
 cc=torch.load(CR/'models'/f'cnn_{h}s_best.pt',map_location='cpu',weights_only=False);bc=torch.load(BR/'models'/f'bilstm_{h}s_best.pt',map_location='cpu',weights_only=False)
 c=EarlyCNN().to(DEV);b=TabularBiLSTM().to(DEV);c.load_state_dict(cc['state_dict']);b.load_state_dict(bc['state_dict'])
 for model in [c,b]:model.requires_grad_(False);model.eval()
 check(f'{h}s_original_training_dependencies',all(before[p]==v for p,v in ck['input_hashes'].items()),'All saved training dependency hashes match, including frozen backbones and datasets.')
 tr=encode('train',h,c,b,cc,bc);diff={}
 for tensor,prefix,dims in [(tr['z'][:,:32],'c',(0,)),(tr['states'],'s',(0,1))]:
  mu=tensor.mean(dims);sd=tensor.std(dims,unbiased=False);sd=torch.where(sd>1e-8,sd,torch.ones_like(sd));diff[prefix+'mean']=float((mu-ck[prefix+'mean']).abs().max());diff[prefix+'scale']=float((sd-ck[prefix+'scale']).abs().max())
 center=float(torch.from_numpy(tr['m'][tr['reg']]).mean());scale=float(torch.from_numpy(tr['m'][tr['reg']]).std(unbiased=False))
 check(f'{h}s_train_fitted_scalers',max(diff.values())==0 and center==ck['target_center'] and scale==ck['target_scale'],diff)
 va=encode('validation',h,c,b,cc,bc);vl,vm,_=attention(va,ck);vp=vl.sigmoid().numpy();vr=va['reg'];loss=float(nn.functional.binary_cross_entropy_with_logits(vl,torch.tensor(va['y'],dtype=torch.float32)))+float(nn.functional.huber_loss(vm[vr],(torch.tensor(va['m'])[vr]-center)/scale))
 best=float('inf');ep=None;stale=0
 for row in ck['history']:
  if row['validation']['total']<best-ck['config']['min_delta']:best=row['validation']['total'];ep=row['epoch'];stale=0
  else:stale+=1
 grid=np.round(np.arange(.10,.901,.01),2);scores=[detection(va['y'],vp,t)['f1'] for t in grid];j=max(range(len(grid)),key=lambda j:(scores[j],-abs(grid[j]-.5),-grid[j]));threshold=float(grid[j])
 check(f'{h}s_validation_selection',ep==ck['best_epoch']=={1:3,2:17,3:32}[h] and abs(loss-ck['best_validation_loss'])<1e-6 and threshold==ck['threshold'] and (stale==7 or len(ck['history'])==35),{'epoch':ep,'epochs_run':len(ck['history']),'loss_delta':loss-ck['best_validation_loss'],'threshold':threshold,'stale_epochs':stale})
 d=encode('test',h,c,b,cc,bc,True);l,z,w=attention(d,ck);p=l.sigmoid().numpy();m=z.numpy()*scale+center;y=d['y'];reg=d['reg'];target=d['m'];digest=hashlib.sha256('\n'.join(d['ids']).encode()).hexdigest()
 check(f'{h}s_common_cohort',len(y)==2414 and reg.sum()==1200 and (~reg).sum()==1214 and digest=='862b467ff6219c5193f4c99257a5971eea395581669302748d006faa78b78601',{'source_order_sha256':digest,'earthquakes':int(reg.sum()),'noise':int((~reg).sum())})
 wd={'shape':list(w.shape),'minimum':float(w.min()),'maximum':float(w.max()),'max_sum_error':float(abs(w.sum(1)-1).max()),'mean_entropy':float((-w*np.log(np.maximum(w,1e-12))).sum(1).mean()),'mean_by_bin':w.mean(0).tolist()}
 check(f'{h}s_temporal_attention',d['states'].shape==(2414,10+10*h,64) and w.shape==(2414,10+10*h) and np.isfinite(w).all() and w.min()>=0 and wd['max_sum_error']<1e-6,wd)
 values={'fixed_05':detection(y,p),'validation_selected':detection(y,p,threshold),'magnitude':magnitude(target[reg],m[reg])};diffs={}
 for k,ref in [('fixed_05',reported[str(h)]['models']['Attention']['fixed_05']),('validation_selected',reported[str(h)]['validation_selected_test']),('magnitude',reported[str(h)]['models']['Attention']['magnitude'])]:
  for name,v in values[k].items():
   if name=='confusion_matrix':check(f'{h}s_{k}_confusion',v==ref[name],v)
   else:diffs[k+'/'+name]=v-ref[name]
 check(f'{h}s_metrics_reproduced',max(abs(v) for v in diffs.values())<1e-6,{'maximum_absolute_difference':max(abs(v) for v in diffs.values())})
 fc=torch.load(FUS/'models'/f'fusion_{h}s_best.pt',map_location='cpu',weights_only=False);fp,fm=fusion(d,fc);fd=detection(y,fp);fmag=magnitude(target[reg],fm[reg]);q=uncertainty(y,target,p,m,fp,fm);ref=reported[str(h)]['paired_comparisons']['Fusion']
 check(f'{h}s_paired_fusion',all(np.allclose(q[k],ref[k],rtol=0,atol=1e-6) for k in q) and abs(fd['f1']-reported[str(h)]['models']['Fusion']['fixed_05']['f1'])<1e-6 and abs(fmag['mae']-reported[str(h)]['models']['Fusion']['magnitude']['mae'])<1e-6,q)
 R['horizons'][h]={'epoch':ep,'threshold':threshold,**values,'metric_differences':diffs,'attention':wd,'fusion':{'detection':fd,'magnitude':fmag},'paired':{'f1_delta':values['fixed_05']['f1']-fd['f1'],'mae_delta':values['magnitude']['mae']-fmag['mae'],**q}}
 print(f'{h}s reproduced: F1={values["fixed_05"]["f1"]:.6f}, MAE={values["magnitude"]["mae"]:.6f}')
check('frozen_backbone_protocol',True,'Inspected training cells: inference-mode cached embeddings; requires_grad False/eval; optimizer receives only AttentionHead parameters; end-of-training tensor equality asserts. Original backbone hashes unchanged. No original training/evaluation cell executed by audit.')
check('observed_window_only',True,'Independent loader supplies exactly200/300/400 samples and20/30/40 feature bins; all hidden states derive only from these inputs. IDs, true magnitudes and arrival metadata are excluded from model inputs. Bidirectionality within observed window allowed; no within-window prefix invariance claim.')
check('validation_only_protocol',True,'Saved source selects checkpoints using validation loss and thresholds using validation F1. All horizons selected before test evaluation. Replayed histories, selected loss and thresholds agree. No retraining/tuning during audit.')
R['checks']['historical_unlogged_actions']={'status':'UNKNOWN','evidence':'Saved source/history/hash evidence supports protocol; no immutable runtime log proves absence of all unrecorded prior tuning. Previously inspected test set is exploratory.'}
changed=[p for p,v in before.items() if sha(Path(p))!=v];check('original_integrity',not changed,{'files':len(before),'changed':changed})
R['verdict']='BLOCKED' if any(v['status']=='FAIL' for v in R['checks'].values()) else 'APPROVED'
with (OUT/'audit_results.json').open('x',encoding='utf-8') as f:json.dump(R,f,indent=2,allow_nan=False)
lines=['# Independent temporal Attention audit','','**Verdict: '+R['verdict']+'** — synthetic oracle-aligned research reporting only.','','## Evidence','','Only original backbone class declarations reused. Attention inference independently implemented with functional linear/tanh/softmax/weighted-sum operations; original Attention forward and evaluation functions not executed. Exact parent joins, frozen checkpoint replay, training-moment recomputation and validation selection replay performed. Existing validated tabular feature-scaler provenance reused; preprocessing audit not repeated.','','| Check | Status | Evidence |','|---|---|---|']
for k,v in R['checks'].items():lines.append(f"| {k} | {v['status']} | {str(v['evidence']).replace('|','/')} |")
lines+=['','## Detection','','| Horizon | Threshold | Accuracy | Precision | Recall | F1 | FPR | ROC-AUC | PR-AUC(AP) |','|---|---:|---:|---:|---:|---:|---:|---:|---:|']
for h,v in R['horizons'].items():
 for key,t in [('fixed_05',.5),('validation_selected',v['threshold'])]:lines.append(f'| {h}s | {t} | '+' | '.join(f'{v[key][k]:.8f}' for k in ['accuracy','precision','recall','f1','false_positive_rate','roc_auc','pr_auc_ap'])+' |')
lines+=['','## Magnitude','','| Horizon | MAE | RMSE | R2 | Signed error | Pred/actual SD | M>=5 N | Underprediction % | Strong bias |','|---|---:|---:|---:|---:|---:|---:|---:|---:|']
for h,v in R['horizons'].items():lines.append(f'| {h}s | '+' | '.join(str(v['magnitude'][k]) for k in ['mae','rmse','r2','bias','std_ratio','strong_n','strong_underprediction_percent','strong_bias'])+' |')
lines+=['','## Attention versus approved fusion','','| Horizon | F1 difference [95% CI] | MAE difference [95% CI] |','|---|---|---|']
for h,v in R['horizons'].items():
 q=v['paired'];lines.append(f"| {h}s | {q['f1_delta']:.8f} {q['f1_difference_ci95']} | {q['mae_delta']:.8f} {q['mae_difference_ci95']} |")
lines+=['','All six intervals include zero: no statistically supported Attention improvement over fusion on either metric at any horizon under this paired bootstrap. 500 paired stratified resamples, seed2026; same eligible earthquake/noise indices per model; regression on all eligible earthquakes independent of detection. Fixed .5 for paired F1. These are descriptive percentile intervals, not equivalence tests, multi-seed evidence or multiplicity-adjusted confirmatory results. Tiny changes must not be interpreted as meaningful gains.','','## Interpretation and limitations','','Epochs3/17/32 reproduce. The3s model reached the35-epoch cap with3 stale epochs; it did not stop by patience7. Best observed validation checkpoint remains epoch32. Attention is genuinely temporal over20/30/40 states with normalized, finite nonnegative weights. Weight entropy, extrema and mean-by-bin are in JSON. Weights are diagnostics, not physical feature importance; bidirectional states mix context across the already observed window.','','Synthetic oracle-P alignment; shared synthetic signal source across branches. No demonstrated operational warning latency, autonomous P detection or real-world EEW readiness. Magnitude compression, M>=5 underprediction and false-positive rates remain concerns. Validation-selected thresholds are reported separately and not compared against fixed-.5 baseline policies. Dataset upstream causality/scaling validation is inherited from approved artifacts, not repeated. Historical unlogged actions and noise-family shortcut removal are not independently established.','','All original dependencies retain their hashes; hashes recorded in JSON. Only audit.ipynb, audit_report.md and audit_results.json created. No training, tuning, new architecture or cleanup.']
with (OUT/'audit_report.md').open('x',encoding='utf-8') as f:f.write('\n'.join(lines)+'\n')
print('Verdict',R['verdict'], 'failures',[k for k,v in R['checks'].items() if v['status']=='FAIL'])


1s reproduced: F1=0.672045, MAE=0.666359
2s reproduced: F1=0.717199, MAE=0.636489
3s reproduced: F1=0.747428, MAE=0.606601
Verdict APPROVED failures []
